### Annotation conversion from XML to GeoJSON (utile per importare annotazioni su QuPath come .geojson objects)
Below is a script that reads the XML files, extracts the annotation information, and converts it into GeoJSON format.

##### Explanation
- Parse XML File: The script uses xml.etree.ElementTree to parse the XML files.
- Extract MicronsPerPixel: Extracts the MicronsPerPixel attribute to scale the coordinates correctly.
- Extract Regions and Vertices: Iterates through the regions and their vertices, converting coordinates to microns.
- Create GeoJSON Features: Constructs GeoJSON features for each region.
- Write GeoJSON File: Saves the features to a GeoJSON file in the specified output directory.

In [ ]:
import xml.etree.ElementTree as ET
import json
import os

DATA_PATH = "src/data"

def convert_xml_to_geojson(xml_path, geojson_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # I file .xml hanno le coordinate già calcolate sui MicronPerPixel mpp,
    # quindi non risulta necessario moltiplicare X e Y di ogni vertice per mpp.

    # mpp = float(root.attrib.get('MicronsPerPixel'))
    # print(mpp) # stampa i MicronPerPixel letti in ogni annotation .xml file

    features = []
    for annotation in root.findall('.//Region'):
        vertices = []
        for vertex in annotation.findall('.//Vertex'):
            x = float(vertex.attrib['X']) # moltiplicare le coordinate per *mpp nel caso in cui la 
            y = float(vertex.attrib['Y']) # il poligono del file .xml abbia una shape errata --> usa QuPath
            vertices.append([x, y])

        # Ensure the polygon is closed by repeating the first vertex at the end if necessary
        if vertices[0] != vertices[-1]:
            vertices.append(vertices[0])

        # Create a feature for each region
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [vertices]
            },
            "properties": {}
        }
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    # Write the GeoJSON to a file
    with open(geojson_path, 'w') as f:
        json.dump(geojson, f, indent=4)

# Directory paths
xml_dir = DATA_PATH + '/annotations'
geojson_dir = DATA_PATH + '/geojson_annotations_converted'

# Create output directory if it doesn't exist
os.makedirs(geojson_dir, exist_ok=True)

# Convert each XML file to GeoJSON
for xml_file in os.listdir(xml_dir):
    if xml_file.endswith('.xml'):
        xml_path = os.path.join(xml_dir, xml_file)
        geojson_path = os.path.join(geojson_dir, os.path.splitext(xml_file)[0] + '.geojson')
        convert_xml_to_geojson(xml_path, geojson_path)
        print(f"Converted {xml_file} to {os.path.splitext(xml_file)[0]}.geojson")
